In [86]:
# ============================================================
# DATA LOADING AND RANDOM TRAINING/VALIDATION/TEST PARTITIONING
# ============================================================

import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import matplotlib.pyplot as plt


In [87]:
RANDOM_STATE = 42
BATCH_SIZE = 16

SPLIT_FILE = "data/processed_features/fixed_split_indices.npz"

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


In [88]:

# ============================================================
# 2. LOAD CURATED BINDINGDB FEATURES
# ============================================================

X_lig = np.load(
    "data/processed_features/X_lig.npy",
    mmap_mode="r"
)

X_prot = np.load(
    "data/processed_features/X_prot.npy",
    mmap_mode="r"
)

y = np.load(
    "data/processed_features/y.npy",
    mmap_mode="r"
)

print("\nDataset:")
print("Ligand shape :", X_lig.shape)
print("Protein shape:", X_prot.shape)
print("Target shape :", y.shape)


Dataset:
Ligand shape : (148111, 1024)
Protein shape: (148111, 320)
Target shape : (148111,)


In [89]:
# ------------------------------------------------------------
# Check alignment
# ------------------------------------------------------------

assert len(X_lig) == len(X_prot) == len(y), (
    "Ligand, protein and target arrays are not aligned."
)

N = len(y)

print("Total samples:", N)

Total samples: 148111


In [90]:
import os

# ============================================================
# 3. CREATE FIXED INDICES
# ============================================================
#
# IMPORTANT:
# We split INDICES rather than X_lig, X_prot and y directly.
#
# This means exactly the same observations can be supplied to
# every machine-learning and deep-learning model.
#
# Final split:
#
# Training   = 64%
# Validation = 16%
# Test       = 20%
#
# ============================================================

if os.path.exists(SPLIT_FILE):

    print("\nLoading existing fixed split...")

    split_data = np.load(SPLIT_FILE)

    train_idx = split_data["train_idx"]
    val_idx = split_data["val_idx"]
    test_idx = split_data["test_idx"]

else:

    print("\nCreating fixed split...")

    all_indices = np.arange(N)

    # --------------------------------------------------------
    # 80% development / 20% test
    # --------------------------------------------------------

    train_val_idx, test_idx = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    # --------------------------------------------------------
    # 80% of development = training
    # 20% of development = validation
    #
    # Overall:
    # training   = 64%
    # validation = 16%
    # test       = 20%
    # --------------------------------------------------------

    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    os.makedirs(
        os.path.dirname(SPLIT_FILE),
        exist_ok=True
    )

    np.savez(
        SPLIT_FILE,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx
    )

    print("Fixed split saved to:")
    print(SPLIT_FILE)



Loading existing fixed split...


In [91]:

# ============================================================
# 4. VERIFY FIXED SPLITS
# ============================================================

print("\nFinal dataset partition:")
print("Training samples  :", len(train_idx))
print("Validation samples:", len(val_idx))
print("Test samples      :", len(test_idx))


Final dataset partition:
Training samples  : 94790
Validation samples: 23698
Test samples      : 29623


In [92]:


# ------------------------------------------------------------
# Verify that no sample appears in multiple partitions
# ------------------------------------------------------------

assert len(
    np.intersect1d(train_idx, val_idx)
) == 0, "Train/validation overlap detected."

assert len(
    np.intersect1d(train_idx, test_idx)
) == 0, "Train/test overlap detected."

assert len(
    np.intersect1d(val_idx, test_idx)
) == 0, "Validation/test overlap detected."

assert (
    len(train_idx) +
    len(val_idx) +
    len(test_idx)
) == N, "Split sizes do not match dataset size."

print("\nSplit verification: PASSED")



Split verification: PASSED


In [93]:

# ============================================================
# 5. APPLY EXACT SAME INDICES TO ALL DATA
# ============================================================

Xl_train = X_lig[train_idx]
Xl_val = X_lig[val_idx]
Xl_test = X_lig[test_idx]

Xp_train = X_prot[train_idx]
Xp_val = X_prot[val_idx]
Xp_test = X_prot[test_idx]

y_train = np.asarray(y[train_idx]).ravel()
y_val = np.asarray(y[val_idx]).ravel()
y_test = np.asarray(y[test_idx]).ravel()

In [94]:
print("\ny_test shape:", y_test.shape)



y_test shape: (29623,)


In [95]:
# ============================================================
# CUSTOM PYTORCH DATASET FOR DRUG–TARGET INTERACTION DATA
# ============================================================

class DrugTargetDataset(Dataset):
    """
    Custom PyTorch Dataset for drug–target affinity regression.

    Each observation consists of:
        1. A numerical ligand representation.
        2. A numerical protein representation.
        3. A continuous binding-affinity target (pKi).

    The Dataset class provides indexed access to individual
    drug–target pairs and converts the stored NumPy arrays into
    PyTorch tensors suitable for model training and evaluation.
    """

    def __init__(self, lig, prot, y):
        """
        Initialise the dataset.

        Parameters
        ----------
        lig : array-like
            Numerical ligand representations corresponding to
            individual drug molecules.

        prot : array-like
            Numerical protein representations corresponding to
            the associated protein targets.

        y : array-like
            Continuous target values representing experimentally
            derived drug–target binding affinity expressed as pKi.
        """

        # Store the ligand representations. Each row corresponds
        # to one drug molecule in a drug–target interaction pair.
        self.lig = lig

        # Store the corresponding protein representations.
        # The ith protein representation must correspond to the
        # same interaction record as the ith ligand representation.
        self.prot = prot

        # Store the continuous affinity values used as the
        # regression targets during supervised model training.
        self.y = y


    def __len__(self):
        """
        Return the total number of drug–target interaction records.

        The number of target values is used because each affinity
        measurement corresponds to exactly one ligand–protein pair.
        """
        return len(self.y)


    def __getitem__(self, idx):
        """
        Retrieve and convert a single drug–target interaction sample.

        Parameters
        ----------
        idx : int
            Index of the requested observation.

        Returns
        -------
        ligand : torch.Tensor
            Ligand representation converted to 32-bit floating point.

        protein : torch.Tensor
            Protein representation converted to an integer tensor.

        target : torch.Tensor
            Experimental pKi value represented as a 32-bit
            floating-point tensor.
        """

        return (
            # Molecular features are represented as floating-point
            # values before being provided to the ligand encoder.
            torch.tensor(
                self.lig[idx],
                dtype=torch.float32
            ),

            # Protein representations are converted to integer
            # tensors because this implementation assumes that the
            # protein input contains tokenised amino-acid indices
            # that are subsequently processed by an embedding layer.
            torch.tensor(
                self.prot[idx],
                dtype=torch.long
            ),

            # The pKi response variable is represented as a
            # floating-point value because affinity prediction is
            # formulated as a continuous regression problem.
            torch.tensor(
                self.y[idx],
                dtype=torch.float32
            ),
        )

In [96]:
# ============================================================
# DATASET INSTANTIATION
# ============================================================

# Construct PyTorch Dataset objects for the training, validation,
# and test partitions. Each dataset maintains the correspondence
# between the ligand representation, protein representation, and
# associated scaled pKi target value.

train_dataset = DrugTargetDataset(
    Xl_train,
    Xp_train,
    y_train
    
)

val_dataset = DrugTargetDataset(
    Xl_val,
    Xp_val,
    y_val
)

test_dataset = DrugTargetDataset(
    Xl_test,
    Xp_test,
    y_test
)


# ============================================================
# DATA LOADER CONFIGURATION
# ============================================================

# Create the DataLoader used during model training.
#
# Samples are processed in mini-batches of size BATCH_SIZE.
# Shuffling is enabled for the training dataset so that the
# ordering of observations changes between epochs. This reduces
# the likelihood that the optimisation process becomes dependent
# on the original ordering of the training observations and
# supports more stable stochastic gradient-based optimisation.

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


# Construct the validation DataLoader.
#
# Shuffling is disabled because the validation partition is not
# used to update model parameters. Maintaining a deterministic
# ordering also facilitates reproducible model evaluation and
# comparison across training epochs.

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Construct the test DataLoader.
#
# The test dataset is evaluated only after model development and
# hyperparameter selection. Therefore, shuffling is unnecessary,
# and the original observation order is retained to support
# reproducible final performance assessment.

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [97]:
# ============================================================
# EXPERIMENTAL AND MODEL CONFIGURATION
# ============================================================

# Define the dimensionality of the ligand representation.
# Each drug molecule is represented using a 1,024-dimensional
# Morgan fingerprint. The fixed dimensionality provides a
# consistent numerical representation for input to the ligand
# encoder.
FP_SIZE = 1024


# Define the maximum protein sequence length considered by the
# model. Protein sequences longer than 300 amino-acid residues
# are truncated during preprocessing, while shorter sequences
# are padded where required to obtain a fixed-length input.
#
# This constraint reduces computational and memory requirements
# and ensures that protein sequences can be processed in
# uniformly shaped mini-batches.
MAX_LEN = 300


# Specify the number of drug–target interaction samples processed
# simultaneously during each optimisation step.
#
# A mini-batch size of 16 provides a compromise between memory
# consumption, computational efficiency, and the stochastic
# behaviour of gradient-based optimisation.
BATCH_SIZE = 16


# Define the maximum number of complete passes through the
# training dataset.
#
# Training is permitted to continue for up to 100 epochs.
# When early stopping is implemented, this value represents an
# upper training limit rather than requiring the model to
# complete all 10 epochs.
EPOCHS = 18

In [98]:
import torch
import torch.nn as nn


# ============================================================
# MULTIMODAL DRUG–TARGET AFFINITY REGRESSION MODEL
# ============================================================

class DTIRegressor(nn.Module):
    """
    Neural network for drug–target affinity prediction.

    The model processes two heterogeneous biological modalities:

    1. Ligand representation:
       A fixed-length Morgan fingerprint describing the
       structural characteristics of the drug molecule.

       Expected shape:
           [batch_size, fingerprint_size]

    2. Protein representation:
       A fixed-length sequence of integer-encoded amino-acid
       tokens.

       Expected shape:
           [batch_size, sequence_length]

    The ligand and protein representations are independently
    encoded into latent feature vectors before being concatenated
    and processed by a fully connected regression head.

    Output:
        Continuous predicted binding affinity expressed as pKi.

        Shape:
            [batch_size]
    """

    def __init__(
        self,
        fingerprint_size=1024,
        protein_vocab_size=22,
        protein_embedding_dim=128,
        padding_idx=0,
        dropout_rate=0.3,
    ):
        """
        Initialise the drug–target affinity regression model.

        Parameters
        ----------
        fingerprint_size : int
            Dimensionality of the Morgan fingerprint used to
            represent each ligand.

        protein_vocab_size : int
            Number of unique amino-acid/token indices available
            in the protein vocabulary, including the padding token.

        protein_embedding_dim : int
            Dimensionality of the trainable amino-acid embedding
            representation.

        padding_idx : int
            Index reserved for sequence padding. The corresponding
            embedding vector is fixed at zero.

        dropout_rate : float
            Dropout probability used for regularisation within
            the ligand, protein, and regression components.
        """

        super().__init__()

        # Store important architectural parameters for subsequent
        # validation of the input tensors.
        self.fingerprint_size = fingerprint_size
        self.padding_idx = padding_idx


        # ========================================================
        # LIGAND ENCODER
        # ========================================================

        # Morgan fingerprints are fixed-length binary vectors in
        # which individual positions correspond to hashed molecular
        # substructures. Unlike sequential data, neighbouring
        # fingerprint positions do not possess an intrinsic spatial
        # or sequential relationship.
        #
        # Consequently, a multilayer perceptron (MLP) is used rather
        # than a convolutional architecture.
        #
        # Dimensionality:
        #
        # 1024 -> 512 -> 256 -> 128

        self.ligand_encoder = nn.Sequential(

            # First nonlinear projection from the original
            # fingerprint space to a lower-dimensional latent space.
            nn.Linear(fingerprint_size, 512),

            # Batch normalisation stabilises the distribution of
            # intermediate activations and can facilitate optimisation.
            nn.BatchNorm1d(512),

            # ReLU introduces non-linearity, allowing the model to
            # represent complex structure–affinity relationships.
            nn.ReLU(),

            # Dropout provides regularisation by randomly deactivating
            # a proportion of hidden units during training.
            nn.Dropout(dropout_rate),

            # Further dimensionality reduction.
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Final ligand representation.
            nn.Linear(256, 128),
            nn.ReLU(),
        )


        # ========================================================
        # PROTEIN TOKEN EMBEDDING
        # ========================================================

        # Integer-encoded amino-acid tokens cannot be processed
        # meaningfully as continuous numerical quantities because
        # their integer values represent categorical identities
        # rather than magnitudes.
        #
        # A trainable embedding layer therefore maps each amino-acid
        # token to a dense 128-dimensional continuous vector.

        self.protein_embedding = nn.Embedding(
            num_embeddings=protein_vocab_size,
            embedding_dim=protein_embedding_dim,
            padding_idx=padding_idx,
        )


        # ========================================================
        # PROTEIN SEQUENCE ENCODER
        # ========================================================

        # A one-dimensional convolutional neural network is used
        # to learn local sequence patterns from the embedded amino-
        # acid representation.
        #
        # This architecture is suitable because neighbouring amino
        # acids possess biologically meaningful sequential context,
        # and convolutional filters can detect recurring local
        # sequence motifs.
        #
        # A kernel size of five allows each convolutional operation
        # to examine local windows spanning five residue positions.

        self.protein_encoder = nn.Sequential(

            # First convolutional layer.
            nn.Conv1d(
                in_channels=protein_embedding_dim,
                out_channels=128,
                kernel_size=5,
                padding=2,
            ),

            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Second convolutional layer provides additional
            # nonlinear transformation of local sequence features.
            nn.Conv1d(
                in_channels=128,
                out_channels=128,
                kernel_size=5,
                padding=2,
            ),

            nn.BatchNorm1d(128),
            nn.ReLU(),
        )


        # ========================================================
        # MULTIMODAL FUSION AND REGRESSION HEAD
        # ========================================================

        # Both modality-specific encoders produce 128-dimensional
        # vectors:
        #
        #     Ligand representation  = 128
        #     Protein representation = 128
        #
        # Concatenation therefore generates a joint
        # 256-dimensional drug–target representation.
        #
        # The fused representation is subsequently transformed
        # through fully connected layers to estimate binding
        # affinity as a single continuous pKi value.

        self.regression_head = nn.Sequential(

            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            # A slightly larger dropout probability is applied
            # immediately after feature fusion to reduce potential
            # overfitting within the joint representation.
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Final regression output containing one scalar affinity
            # prediction for each drug–target pair.
            nn.Linear(128, 1),
        )


        # Apply explicit parameter initialisation after all model
        # components have been constructed.
        self._initialize_weights()


    # ============================================================
    # PARAMETER INITIALISATION
    # ============================================================

    def _initialize_weights(self):
        """
        Initialise trainable model parameters.

        Kaiming normal initialisation is applied to linear and
        convolutional layers because these layers are followed
        predominantly by ReLU activation functions.

        Protein embedding weights are initialised from a small
        normal distribution. The embedding corresponding to the
        padding token is explicitly maintained as a zero vector.
        """

        for module in self.modules():

            # ----------------------------------------------------
            # Fully connected layers
            # ----------------------------------------------------
            if isinstance(module, nn.Linear):

                nn.init.kaiming_normal_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)


            # ----------------------------------------------------
            # One-dimensional convolutional layers
            # ----------------------------------------------------
            elif isinstance(module, nn.Conv1d):

                nn.init.kaiming_normal_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)


            # ----------------------------------------------------
            # Amino-acid embedding layer
            # ----------------------------------------------------
            elif isinstance(module, nn.Embedding):

                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.02,
                )

                # Ensure that the padding token does not contribute
                # a learned representation.
                if module.padding_idx is not None:

                    with torch.no_grad():
                        module.weight[
                            module.padding_idx
                        ].zero_()


    # ============================================================
    # MASKED GLOBAL MAX POOLING
    # ============================================================

    def masked_max_pool(
        self,
        protein_features,
        protein_mask,
    ):
        """
        Perform global max pooling across the protein sequence
        while excluding padded positions.

        Parameters
        ----------
        protein_features : torch.Tensor
            Protein feature tensor with dimensions:

            [batch_size, channels, sequence_length]

        protein_mask : torch.Tensor
            Boolean mask with dimensions:

            [batch_size, sequence_length]

            True values identify genuine amino-acid positions,
            whereas False values identify padding.

        Returns
        -------
        torch.Tensor
            Fixed-length protein representation with dimensions:

            [batch_size, channels]
        """

        # Extend the mask with a channel dimension so that it can
        # be broadcast across all convolutional feature channels.
        #
        # [batch, sequence_length]
        #            ->
        # [batch, 1, sequence_length]

        protein_mask = protein_mask.unsqueeze(1)


        # Assign a very small floating-point value to padded
        # positions before max pooling. This prevents artificial
        # padding values from being selected as sequence maxima.
        protein_features = protein_features.masked_fill(
            ~protein_mask,
            torch.finfo(protein_features.dtype).min,
        )


        # Perform global maximum pooling over the sequence
        # dimension.
        #
        # [batch, 128, sequence_length]
        #            ->
        # [batch, 128]

        pooled_features = protein_features.max(dim=2).values


        # Identify pathological cases in which a protein sequence
        # contains no valid amino-acid tokens and consists entirely
        # of padding.
        invalid_sequences = ~protein_mask.any(dim=2)


        # Replace representations of completely padded sequences
        # with zero vectors to prevent extreme negative values from
        # propagating through the network.
        if invalid_sequences.any():

            pooled_features = pooled_features.masked_fill(
                invalid_sequences,
                0.0,
            )


        return pooled_features


    # ============================================================
    # FORWARD PROPAGATION
    # ============================================================

    def forward(self, ligand, protein):
        """
        Perform forward propagation through the multimodal model.

        Parameters
        ----------
        ligand : torch.Tensor
            Morgan fingerprint tensor with dimensions:

            [batch_size, fingerprint_size]

        protein : torch.Tensor
            Integer-encoded protein sequence tensor with dimensions:

            [batch_size, sequence_length]

        Returns
        -------
        torch.Tensor
            Predicted pKi values with dimensions:

            [batch_size]
        """


        # ========================================================
        # INPUT VALIDATION
        # ========================================================

        # Verify that ligand inputs follow the required
        # two-dimensional batch representation.
        if ligand.ndim != 2:

            raise ValueError(
                "Ligand input must have shape "
                "[batch_size, fingerprint_size]."
            )


        # Verify consistency between the observed fingerprint
        # dimensionality and the model configuration.
        if ligand.size(1) != self.fingerprint_size:

            raise ValueError(
                f"Expected {self.fingerprint_size} ligand features, "
                f"but received {ligand.size(1)}."
            )


        # Protein input must contain one token sequence per sample.
        if protein.ndim != 2:

            raise ValueError(
                "Protein input must have shape "
                "[batch_size, sequence_length]."
            )


        # Ensure that each ligand corresponds to exactly one
        # protein sequence within the same mini-batch.
        if ligand.size(0) != protein.size(0):

            raise ValueError(
                "Ligand and protein inputs must have the same batch size."
            )


        # ========================================================
        # DATA TYPE STANDARDISATION
        # ========================================================

        # Morgan fingerprints are represented as floating-point
        # tensors for processing by fully connected neural layers.
        ligand = ligand.float()

        # Protein token indices must use integer type because they
        # are passed to the nn.Embedding lookup layer.
        protein = protein.long()


        # ========================================================
        # LIGAND FEATURE EXTRACTION
        # ========================================================

        # Convert the original 1,024-dimensional Morgan fingerprint
        # into a learned 128-dimensional ligand representation.
        #
        # [batch, 1024] -> [batch, 128]

        ligand_features = self.ligand_encoder(ligand)


        # ========================================================
        # PROTEIN FEATURE EXTRACTION
        # ========================================================

        # Generate a Boolean mask distinguishing valid amino-acid
        # positions from padded positions.
        protein_mask = protein.ne(self.padding_idx)


        # Convert integer amino-acid tokens into continuous,
        # trainable embedding vectors.
        #
        # [batch, sequence_length]
        #            ->
        # [batch, sequence_length, embedding_dim]

        protein_features = self.protein_embedding(protein)


        # PyTorch Conv1d expects the channel dimension before the
        # sequence dimension. The embedding tensor is therefore
        # rearranged accordingly.
        #
        # [batch, sequence_length, embedding_dim]
        #                    ->
        # [batch, embedding_dim, sequence_length]

        protein_features = protein_features.permute(
            0,
            2,
            1,
        )


        # Apply the convolutional sequence encoder to learn
        # local amino-acid patterns and motifs.
        #
        # Output:
        # [batch, 128, sequence_length]

        protein_features = self.protein_encoder(
            protein_features
        )


        # Aggregate variable-length sequence information into a
        # fixed 128-dimensional protein representation while
        # excluding padded positions.
        #
        # [batch, 128, sequence_length]
        #            ->
        # [batch, 128]

        protein_features = self.masked_max_pool(
            protein_features,
            protein_mask,
        )


        # ========================================================
        # MULTIMODAL FEATURE FUSION
        # ========================================================

        # Concatenate the learned ligand and protein representations
        # along the feature dimension.
        #
        # Ligand:  [batch, 128]
        # Protein: [batch, 128]
        #
        # Combined:
        #          [batch, 256]

        combined_features = torch.cat(
            [
                ligand_features,
                protein_features,
            ],
            dim=1,
        )


        # ========================================================
        # AFFINITY REGRESSION
        # ========================================================

        # Process the fused drug–target representation through
        # the regression head to generate a single continuous
        # affinity prediction for each interaction.
        #
        # [batch, 256] -> [batch, 1]

        predictions = self.regression_head(
            combined_features
        )


        # Remove the final singleton dimension:
        #
        # [batch, 1] -> [batch]
        #
        # This shape is convenient for comparison with the
        # one-dimensional ground-truth pKi tensor.

        return predictions.squeeze(-1)

In [99]:
# ============================================================
# MODEL INITIALISATION
# ============================================================

# Instantiate the proposed drug–target affinity regression model.
#
# The ligand branch receives 1,024-dimensional Morgan fingerprints,
# while the protein branch processes integer-encoded amino-acid
# sequences using a trainable embedding layer followed by a
# one-dimensional convolutional encoder.
#
# The complete model is transferred to the selected computational
# device (Apple MPS when available, otherwise CPU).

model = DTIRegressor(
    # fingerprint_size=1024,
    # protein_vocab_size=22,
    # # protein_embedding_dim=128,
    # padding_idx=0,
    # dropout_rate=0.3,
).to(device)


# Display the instantiated architecture to verify the model
# configuration and individual network components before training.
print(model)


# ============================================================
# LOSS FUNCTION
# ============================================================

# Mean squared error (MSE) was considered as a conventional
# regression objective:
#
# loss_fn = nn.MSELoss()
#
# However, Huber loss is used as the optimisation objective in
# the final configuration. Huber loss behaves approximately
# quadratically for small prediction errors and linearly for
# larger errors. Consequently, it is less sensitive to extreme
# residuals than conventional MSE while remaining differentiable
# and suitable for gradient-based optimisation.
#
# delta=1.0 defines the transition point between the quadratic
# and linear components of the Huber loss function.

loss_fn = nn.HuberLoss(delta=1.0)


# ============================================================
# OPTIMISER
# ============================================================

# AdamW is used to optimise the trainable model parameters.
#
# AdamW extends the Adam optimisation algorithm by applying
# decoupled weight decay, allowing regularisation to be controlled
# independently from the adaptive gradient updates.
#
# Learning rate:
#     3 × 10^-4
#
# Weight decay:
#     1 × 10^-4
#
# The weight-decay term provides additional regularisation and
# can reduce overfitting by discouraging excessively large model
# parameters.

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-3
)


# ============================================================
# ADAPTIVE LEARNING-RATE SCHEDULER
# ============================================================

# ReduceLROnPlateau dynamically decreases the learning rate when
# validation performance stops improving.
#
# mode="min":
#     Lower validation loss represents improved performance.
#
# factor=0.5:
#     The current learning rate is multiplied by 0.5 whenever
#     the reduction criterion is satisfied.
#
# patience=5:
#     The scheduler waits for five validation evaluations without
#     sufficient improvement before reducing the learning rate.
#
# This strategy allows relatively large optimisation steps during
# the earlier stages of training while enabling finer parameter
# updates once validation performance begins to plateau.

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)


# ============================================================
# EARLY-STOPPING CONFIGURATION
# ============================================================

# Initialise the best observed validation loss to positive
# infinity. Any finite validation loss obtained during the first
# training epoch will therefore represent an improvement.

best_val_loss = float("inf")


# Define the early-stopping patience.
#
# Training will be terminated if the validation loss fails to
# improve for 15 consecutive epochs. This prevents unnecessary
# optimisation once generalisation performance ceases to improve
# and helps reduce the risk of overfitting.

patience = 15


# Counter used to record the number of consecutive epochs without
# improvement in validation loss. The counter is reset to zero
# whenever a new best validation loss is observed.

counter = 0

DTIRegressor(
  (ligand_encoder): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): ReLU()
  )
  (protein_embedding): Embedding(22, 128, padding_idx=0)
  (protein_encoder): Sequential(
    (0): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(128, eps=1e-05, momentum=0

In [100]:
import pandas as pd

# Counter recording consecutive epochs without improvement.
counter = 0
EPOCHS = 100

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "learning_rate": [],
}


# ============================================================
# MODEL TRAINING
# ============================================================

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAINING PHASE
    # --------------------------------------------------------

    # Set the network to training mode. This activates training-
    # specific behaviour in layers such as Dropout and
    # BatchNorm1d.
    model.train()

    # Accumulate mini-batch losses to calculate the mean training
    # loss for the current epoch.
    running_loss = 0.0

    for ligand, protein, label in train_loader:

        # Transfer the current mini-batch to the selected device.
        ligand = ligand.to(device)
        protein = protein.to(device)
        label = label.to(device)

        # ----------------------------------------------------
        # FORWARD PROPAGATION
        # ----------------------------------------------------

        # Generate predicted affinity values for the current
        # drug–target interaction mini-batch.
        outputs = model(
            ligand,
            protein
        )

        # Calculate the discrepancy between predicted and observed
        # affinity values using Huber loss.
        loss = loss_fn(
            outputs,
            label
        )


        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        # Clear gradients accumulated during the previous
        # optimisation step.
        optimizer.zero_grad()

        # Compute gradients of the loss with respect to all
        # trainable parameters.
        loss.backward()


        # ----------------------------------------------------
        # GRADIENT CLIPPING
        # ----------------------------------------------------

        # Limit the global gradient norm to a maximum value of 5.
        # Gradient clipping can improve numerical stability by
        # preventing excessively large gradient updates during
        # optimisation.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )


        # ----------------------------------------------------
        # PARAMETER UPDATE
        # ----------------------------------------------------

        # Update the model parameters using the AdamW optimiser.
        optimizer.step()

        # Accumulate the loss from the current mini-batch.
        running_loss += loss.item()


    # Calculate the mean training loss across all mini-batches
    # processed during the current epoch.
    running_loss /= len(train_loader)


    # ========================================================
    # VALIDATION PHASE
    # ========================================================

    # Switch the network to evaluation mode. This disables
    # dropout and causes batch-normalisation layers to use their
    # stored running statistics rather than updating them.
    model.eval()

    # Accumulate validation loss across all validation batches.
    val_loss = 0.0


    # Gradient computation is unnecessary during validation
    # because no parameter updates are performed.
    with torch.no_grad():

        for ligand, protein, affinity in val_loader:

            # Transfer validation tensors to the selected device.
            ligand = ligand.to(device)
            protein = protein.to(device)
            affinity = affinity.to(device)


            # Generate affinity predictions for the validation
            # drug–target pairs.
            prediction = model(
                ligand,
                protein
            )


            # Calculate validation loss using the same regression
            # objective employed during model training.
            loss = loss_fn(
                prediction,
                affinity
            )

            # Accumulate batch-level validation loss.
            val_loss += loss.item()


    # Calculate the mean validation loss across the complete
    # validation partition.
    val_loss /= len(val_loader)


    # ========================================================
    # LEARNING-RATE ADAPTATION
    # ========================================================

    # Supply validation loss to ReduceLROnPlateau. If validation
    # performance fails to improve for the configured scheduler
    # patience period, the learning rate is reduced.
    scheduler.step(val_loss)


    # Retrieve the current learning rate for training diagnostics.
    current_lr = optimizer.param_groups[0]["lr"]

    # ========================================================
    # RECORD TRAINING HISTORY
    # ========================================================

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(running_loss)
    history["val_loss"].append(val_loss)
    history["learning_rate"].append(current_lr)

    # --------------------------------------------------------
    # SAVE TRAINING HISTORY TO CSV
    # --------------------------------------------------------

    # The CSV file is updated after every epoch. Consequently,
    # training information is retained even if training is
    # interrupted before all epochs are completed.
    

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        "dti_training_history.csv",
        index=False
    )



    # ========================================================
    # TRAINING PROGRESS
    # ========================================================

    # Report epoch-level training and validation losses together
    # with the current optimiser learning rate.
    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {running_loss:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"LR: {current_lr:.6f}"
        
    )


    # ========================================================
    # MODEL CHECKPOINTING AND EARLY STOPPING
    # ========================================================

    # Determine whether the current epoch produced a lower
    # validation loss than any previous epoch.
    if val_loss < best_val_loss:

        # Update the best validation performance obtained so far.
        best_val_loss = val_loss

        # best_epoch is updated to reflect the epoch number corresponding
        # to the lowest validation loss observed during training.
        best_epoch = epoch + 1

        # Reset the early-stopping counter because validation
        # performance has improved.
        counter = 0


        # Save the model parameters corresponding to the best
        # observed validation loss. This ensures that subsequent
        # test evaluation uses the model with the strongest
        # validation performance rather than simply the model
        # obtained during the final training epoch.
        torch.save(
            model.state_dict(),
            "best_dti_model.pt"
        )

        print("✓ Best model saved")


    else:

        # Increment the number of consecutive epochs without
        # validation improvement.
        counter += 1

        print(
            f"Early stopping counter: "
            f"{counter}/{patience}"
        )
        

        # Stop training when validation performance has failed to
        # improve for the predefined number of epochs.
        if counter >= patience:

            print("Early stopping triggered")

            break

Epoch [1/100] Train Loss: 0.9378 Val Loss: 2.2210 LR: 0.000300
✓ Best model saved
Epoch [2/100] Train Loss: 0.7667 Val Loss: 0.9414 LR: 0.000300
✓ Best model saved
Epoch [3/100] Train Loss: 0.7058 Val Loss: 0.6786 LR: 0.000300
✓ Best model saved
Epoch [4/100] Train Loss: 0.6664 Val Loss: 0.6189 LR: 0.000300
✓ Best model saved
Epoch [5/100] Train Loss: 0.6307 Val Loss: 0.5585 LR: 0.000300
✓ Best model saved
Epoch [6/100] Train Loss: 0.6039 Val Loss: 0.5945 LR: 0.000300
Early stopping counter: 1/15
Epoch [7/100] Train Loss: 0.5814 Val Loss: 0.5346 LR: 0.000300
✓ Best model saved
Epoch [8/100] Train Loss: 0.5600 Val Loss: 0.5329 LR: 0.000300
✓ Best model saved
Epoch [9/100] Train Loss: 0.5393 Val Loss: 0.5376 LR: 0.000300
Early stopping counter: 1/15
Epoch [10/100] Train Loss: 0.5196 Val Loss: 0.5254 LR: 0.000300
✓ Best model saved
Epoch [11/100] Train Loss: 0.5065 Val Loss: 0.5189 LR: 0.000300
✓ Best model saved
Epoch [12/100] Train Loss: 0.4913 Val Loss: 0.5608 LR: 0.000300
Early stoppi

KeyboardInterrupt: 

In [101]:
model.eval()

DTIRegressor(
  (ligand_encoder): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): ReLU()
  )
  (protein_embedding): Embedding(22, 128, padding_idx=0)
  (protein_encoder): Sequential(
    (0): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(128, eps=1e-05, momentum=0

In [102]:
def evaluate_split(model, loader, criterion, device):
    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():
        for ligand, protein, y in loader:
            ligand = ligand.to(device)
            protein = protein.to(device)
            y = y.to(device)

            output = model(ligand, protein)

            predictions.extend(output.cpu().numpy())
            targets.extend(y.cpu().numpy())

    predictions = np.asarray(predictions)
    targets = np.asarray(targets)

    rmse = np.sqrt(mean_squared_error(targets, predictions))
    mae = mean_absolute_error(targets, predictions)
    r2 = r2_score(targets, predictions)
    pearson = np.corrcoef(targets, predictions)[0, 1]
    # ci = concordance_index(
    #         targets,
    #         predictions
    #     )

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Pearson": pearson,
        # "CI": ci
    }
# Train: {'RMSE': np.float64(0.42871898391575997), 'MAE': 0.3087441921234131, 'R2': 0.8162000179290771, 'Pearson': np.float64(0.9106309977820247)}
# Validation: {'RMSE': np.float64(0.7877771086802807), 'MAE': 0.5996342897415161, 'R2': 0.3778798580169678, 'Pearson': np.float64(0.6261924654509952)}
# Test: {'RMSE': np.float64(0.7947358984690636), 'MAE': 0.6037222743034363, 'R2': 0.37590640783309937, 'Pearson': np.float64(0.6248231585487828)}

train_results = evaluate_split(model, train_loader, loss_fn, device)
val_results = evaluate_split(model, val_loader, loss_fn, device)
test_results = evaluate_split(model, test_loader, loss_fn, device)

print("Train:", train_results)
print("Validation:", val_results)
print("Test:", test_results)


Train: {'RMSE': np.float64(0.8606665875616808), 'MAE': 0.6410612463951111, 'R2': 0.6391724348068237, 'Pearson': np.float64(0.8015044378623368)}
Validation: {'RMSE': np.float64(1.145951519275189), 'MAE': 0.8839368224143982, 'R2': 0.3587462902069092, 'Pearson': np.float64(0.6059492835015945)}
Test: {'RMSE': np.float64(1.1567943425761709), 'MAE': 0.8894177675247192, 'R2': 0.3559105396270752, 'Pearson': np.float64(0.6034196542493158)}
